## Sistema inteligente de control de tráfico que optimice los tiempos de semáforos y detecte congestiones en tiempo real mediante técnicas de minería de datos y visión por computadora
jupyter nbconvert --clear-output YOLO.ipynb

## Importar librerías y cargar el modelo YOLO

In [ ]:
from traffic_system import TrafficSystem
import cv2
from pymongo import MongoClient
from datetime import datetime
import os
import gridfs
import numpy as np

traffic_system = TrafficSystem()
traffic_system.load_model('yolov8m.pt')

## Procesar video, analizar frames y guardar resultados en MongoDB + GridFS

In [ ]:
def procesar_video_y_guardar_en_mongo_gridfs(video_path, traffic_system, frame_interval=30, descripcion=""):
    client = MongoClient("mongodb://localhost:27017/")
    db = client['trafico']
    fs = gridfs.GridFS(db)
    videos_col = db['Videos']
    frames_col = db['VideosFrames']

    # Insertar el video y obtener el video_id automáticamente
    video_doc = {
        "nombre": os.path.basename(video_path),
        "ruta": video_path,
        "fecha_subida": datetime.now(),
        "descripcion": descripcion,
        "total_frames": int(cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FRAME_COUNT))
    }
    video_id = videos_col.insert_one(video_doc).inserted_id

    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    saved_count = 0
    estadisticas_autos = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % frame_interval == 0:
            # Procesar el frame con YOLO
            processed_image, detections = traffic_system.detect_objects_from_array(frame)

            # Calcular métricas usando el método analyze_traffic de TrafficSystem
            metrics = traffic_system.analyze_traffic(detections, saved_count, video_id)

            estadisticas_autos.append(metrics.get('by_class', {}).get('car', 0))

            # Guardar imagen procesada en GridFS
            _, buffer = cv2.imencode('.jpg', processed_image)
            file_id = fs.put(buffer.tobytes(), filename=f'processed_{saved_count:04d}.jpg')

            # Guardar en MongoDB con métricas
            frame_doc = {
                "video_id": video_id,
                "frame_num": saved_count,
                "gridfs_id": file_id,
                "detecciones": detections,
                "metricas": metrics
            }
            frames_col.insert_one(frame_doc)

            saved_count += 1

        frame_count += 1

    cap.release()
    print(f"Video y {saved_count} frames procesados y almacenados en MongoDB (GridFS).")
    return video_id, saved_count, estadisticas_autos

## Reconstruir video procesado desde GridFS y registrar en MongoDB

In [ ]:
def reconstruir_video_procesado_gridfs(video_id, output_video_path, descripcion="", parametros=None, estadisticas=None):
    client = MongoClient("mongodb://localhost:27017/")
    db = client['trafico']
    fs = gridfs.GridFS(db)
    frames_col = db['VideosFrames']
    videos_reconstruidos_col = db['VideosReconstruidos']

    # Obtener todos los frames de este video, ordenados
    frames_cursor = frames_col.find({"video_id": video_id}).sort("frame_num", 1)
    frames_list = list(frames_cursor)
    if not frames_list:
        print("No se encontraron frames para este video.")
        return

    # Leer el primer frame para obtener tamaño y fps
    first_img_bytes = fs.get(frames_list[0]['gridfs_id']).read()
    nparr = np.frombuffer(first_img_bytes, np.uint8)
    frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    height, width, _ = frame.shape
    fps = 10  # Ajusta según tu video original

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    for frame_doc in frames_list:
        img_bytes = fs.get(frame_doc['gridfs_id']).read()
        nparr = np.frombuffer(img_bytes, np.uint8)
        frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        if frame is not None:
            out.write(frame)
    out.release()
    print(f"Video reconstruido guardado en: {output_video_path}")

    # Guardar en MongoDB
    doc = {
        "video_id": video_id,
        "ruta_video_reconstruido": output_video_path,
        "fecha_creacion": datetime.now(),
        "descripcion": descripcion,
        "parametros": parametros or {},
        "estadisticas": estadisticas or {}
    }
    videos_reconstruidos_col.insert_one(doc)
    print("Video reconstruido registrado en MongoDB.")

## Procesar el video y reconstruirlo automáticamente

In [ ]:
video_path = 'videos/tradffic2.mp4'
frame_interval = 30
descripcion = "Video de tráfico urbano"

video_id, frame_count, estadisticas_autos = procesar_video_y_guardar_en_mongo_gridfs(
    video_path, traffic_system, frame_interval, descripcion
)

output_video_path = f'frames_analizados/traffic1_procesado.mp4'
parametros = {"frame_interval": frame_interval, "modelo": "yolov8n.pt"}
estadisticas = {
    "frames_procesados": frame_count,
    "promedio_autos": sum(estadisticas_autos) / len(estadisticas_autos) if estadisticas_autos else 0
}

reconstruir_video_procesado_gridfs(
    video_id, output_video_path,
    descripcion="Reconstrucción con detecciones YOLOv8 (GridFS)",
    parametros=parametros, estadisticas=estadisticas
)

## Crear un GIF animado desde GridFS

In [ ]:
import imageio

client = MongoClient("mongodb://localhost:27017/")
db = client['trafico']
fs = gridfs.GridFS(db)
frames_col = db['VideosFrames']

frames_cursor = frames_col.find({"video_id": video_id}).sort("frame_num", 1)
frames_list = list(frames_cursor)

frames = []
for frame_doc in frames_list:
    img_bytes = fs.get(frame_doc['gridfs_id']).read()
    nparr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if img is not None:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        frames.append(img_rgb)

gif_path = 'frames_analizados/detections_gridfs.gif'
if frames:
    imageio.mimsave(gif_path, frames, fps=10,loop=0)
    print(f"GIF guardado en: {gif_path}")
else:
    print("No se encontraron frames en GridFS para crear el GIF.")

## Visualizar el GIF en el notebook

In [ ]:
from IPython.display import Image, display
import os

gif_path = 'frames_analizados/detections_gridfs.gif'
if os.path.exists(gif_path):
    display(Image(filename=gif_path))
else:
    print("El archivo GIF no existe. Verifica que se haya creado correctamente.")